# 7. Counting spiritual and pastoral books with an open language model

I classify each edition with an open-weights model (Qwen3 8B, Apache 2.0, run with Ollama) from its title, its author and, when
the author is linked to Wikidata, a short note on the author (description, occupation, religious order).

1. **Pilot:** 200 titles I labeled by hand in my Jesuit missionary-libraries project (`data/llm_pilot_titles_bio.tsv`), to measure
   precision and recall.
2. **Catalog sample:** 5,000 USTC and BNE records, stratified by the keyword lexicon (`data/llm_catalog_sample.tsv`), to estimate
   how many spiritual and pastoral editions the whole catalog holds.

I wrote it for Google Colab with a T4 GPU: upload both files to `/content` before running. These are the outputs of my run;
notebook 8 reads the answers from `data/llm_catalog_sample_results.tsv`.

---

# 7. Conteo de libros espirituales y pastorales con un modelo de lenguaje abierto

Clasifico cada edición con un modelo de pesos abiertos (Qwen3 8B, Apache 2.0, corrido con Ollama) a partir del título, el autor y,
cuando el autor está enlazado con Wikidata, una nota breve sobre él (descripción, ocupación, orden religiosa).

1. **Piloto:** 200 títulos que etiqueté a mano en mi proyecto de bibliotecas misionales jesuitas (`data/llm_pilot_titles_bio.tsv`),
   para medir la precisión y la exhaustividad (recall).
2. **Muestra del catálogo:** 5.000 registros del USTC y la BNE, estratificados por el léxico (`data/llm_catalog_sample.tsv`), para
   estimar cuántas ediciones espirituales y pastorales tiene el catálogo completo.

Lo escribí para Google Colab con una GPU T4: antes de correrlo hay que subir los dos archivos a `/content`. Estas son las salidas
de mi corrida; el notebook 8 lee las respuestas de `data/llm_catalog_sample_results.tsv`.

In [1]:
# Celda 1: Paths and model
PILOT_FILE = "/content/llm_pilot_titles_bio.tsv"
SAMPLE_FILE = "/content/llm_catalog_sample.tsv"
SAMPLE_RESULTS_FILE = "/content/llm_catalog_sample_results.tsv"
MODEL = "qwen3:8b"
MODELS_DIR = "/content/ollama_models"

In [2]:
# Celda 2: Install Ollama (zstd is needed to unpack it)
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# Celda 3: Start the Ollama server in its own session, so interrupting a cell does not stop it
import os, subprocess, time
os.environ["OLLAMA_MODELS"] = MODELS_DIR
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)
time.sleep(5)
!ollama --version

ollama version is 0.40.13


In [4]:
# Celda 4: Download the model (about 5 GB)
!ollama pull {MODEL}

In [5]:
# Celda 5: Load the pilot and the catalog sample
import pandas as pd
pilot = pd.read_csv(PILOT_FILE, sep="\t", dtype=str).fillna("")
sample = pd.read_csv(SAMPLE_FILE, sep="\t", dtype={"id": str}).fillna("")
print(len(pilot), len(sample))
sample.groupby("lexicon_class")["weight"].agg(["size", "first"])

200 5000


,size,first
lexicon_class,,
none,2500,52.8272
other_pastoral,1250,16.5360
spiritual,1250,8.3448


In [6]:
# Celda 6: Instructions for the model, from data/labeling_criteria.md plus a non-religious category
INSTRUCTIONS = """Clasificás ediciones españolas de 1500-1800 a partir del título, el autor y, si la hay, una nota sobre el autor.
Elegí UNA categoría:
- espiritual: oración, meditación, ejercicios espirituales, devoción, novenas, rosarios, vía crucis, perfección, ascética y mística, desengaños, preparación para la muerte o la comunión, vida cristiana, consuelos, poesía devota (Granada, Kempis, Teresa, Nieremberg...).
- predicable: sermones, oraciones evangélicas, fúnebres o panegíricas, sermonarios, cuaresmas, pláticas.
- historia_sacra: vidas de santos y venerables, milagros, imágenes y santuarios, crónicas de órdenes, canonizaciones, historia eclesiástica.
- teologia_moral: casos de conciencia, sumas morales, confesionarios, instrucciones de confesores.
- catecismo: catecismos, doctrinas cristianas, cartillas.
- teologia: dogmática, controversia, tratados doctrinales.
- biblia: Biblias, salmos, comentarios bíblicos.
- liturgia: misales, breviarios, oficios, horas, ceremoniales, villancicos.
- gobierno_religioso: reglas, constituciones, sínodos, edictos, bulas, indulgencias, pleitos y memoriales de conventos, Inquisición.
- no_religioso: literatura, teatro, novela, derecho, historia civil, ciencia, gramática, política.
La nota sobre el autor ayuda cuando el título es poco claro, pero lo que decide es el título.
Si el título es ambiguo, elegí el uso principal que sugiere."""

CATEGORIES = ["espiritual", "predicable", "historia_sacra", "teologia_moral", "catecismo",
              "teologia", "biblia", "liturgia", "gobierno_religioso", "no_religioso"]

In [7]:
# Celda 7: Ask the model for one category; the answer is forced to be one of CATEGORIES
import json, requests

def classify(author, title, author_bio):
    question = "Autor: " + author + "\nTítulo: " + title
    if author_bio:
        question = question + "\nNota sobre el autor: " + author_bio
    answer_format = {"type": "object",
                     "properties": {"category": {"type": "string", "enum": CATEGORIES}},
                     "required": ["category"]}
    reply = requests.post("http://localhost:11434/api/chat", json={
        "model": MODEL,
        "messages": [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": question}],
        "format": answer_format,
        "think": False,
        "stream": False,
        "options": {"temperature": 0},
    })
    return json.loads(reply.json()["message"]["content"])["category"]

classify("Granada, Luis de", "Guia de pecadores", "")   # Should be "espiritual"

'espiritual'

In [8]:
# Celda 8: Turn the ten categories into the three classes of the count
# "predicable" -> "other_pastoral", "biblia" -> "not_pastoral":
OTHER_PASTORAL = ["predicable", "historia_sacra", "teologia_moral", "catecismo"]

def llm_class(category):
    if category == "espiritual":
        return "spiritual"
    if category in OTHER_PASTORAL:
        return "other_pastoral"
    return "not_pastoral"

In [9]:
# Celda 9: Pilot, with and without the author note
start = time.time()
pilot["llm_class_title"] = [llm_class(classify(r["author"], r["title"], "")) for i, r in pilot.iterrows()]
pilot["llm_class_bio"] = [llm_class(classify(r["author"], r["title"], r["author_bio"])) for i, r in pilot.iterrows()]
print(f"{2 * len(pilot)} calls in {time.time() - start:.0f} s")

for method in ["lexicon_class", "llm_class_title", "llm_class_bio"]:
    for name, classes in [("spiritual", ["spiritual"]), ("pastoral", ["spiritual", "other_pastoral"])]:
        true = pilot["true_class"].isin(classes)
        marked = pilot[method].isin(classes)
        hits = (true & marked).sum()
        print(f"{method:16s} {name:9s}: precision {hits / marked.sum():.2f} | recall {hits / true.sum():.2f}")

400 calls in 256 s
lexicon_class    spiritual: precision 0.76 | recall 0.68
lexicon_class    pastoral : precision 0.88 | recall 0.86
llm_class_title  spiritual: precision 0.97 | recall 0.72
llm_class_title  pastoral : precision 0.93 | recall 0.89
llm_class_bio    spiritual: precision 1.00 | recall 0.64
llm_class_bio    pastoral : precision 0.94 | recall 0.87


In [11]:
# Celda 10: Classify the catalog sample, saving every 500 records in case the session ends
# Change USE_BIO to False if the pilot shows that the author note does not help:
USE_BIO = False
start = time.time()
llm_categories = []
for i, row in sample.iterrows():
    llm_categories.append(classify(row["author"], row["title"], row["author_bio"] if USE_BIO else ""))
    if len(llm_categories) % 500 == 0:
        print(len(llm_categories), f"{time.time() - start:.0f} s")
        sample.head(len(llm_categories)).assign(llm_category=llm_categories).to_csv(SAMPLE_RESULTS_FILE, sep="\t", index=False)
sample["llm_category"] = llm_categories
sample["llm_class"] = sample["llm_category"].map(llm_class)
sample.to_csv(SAMPLE_RESULTS_FILE, sep="\t", index=False)

500 328 s
1000 656 s
1500 983 s
2000 1309 s
2500 1634 s
3000 1962 s
3500 2289 s
4000 2614 s
4500 2942 s
5000 3272 s


In [12]:
# Celda 11: Estimated totals for the whole catalog, with a 95% interval
# Each stratum (lexicon class) has N records and n sampled; the share p found in the sample is scaled to N:
import math
for name, classes in [("spiritual", ["spiritual"]), ("pastoral", ["spiritual", "other_pastoral"])]:
    total = 0
    variance = 0
    for stratum, rows in sample.groupby("lexicon_class"):
        n = len(rows)
        N = n * rows["weight"].iloc[0]
        p = rows["llm_class"].isin(classes).mean()
        total = total + N * p
        variance = variance + N * N * p * (1 - p) / n
    margin = 1.96 * math.sqrt(variance)
    print(f"{name:9s}: LLM estimate {total:,.0f} (95% interval {total - margin:,.0f} to {total + margin:,.0f})")

spiritual: LLM estimate 9,764 (95% interval 8,956 to 10,573)
pastoral : LLM estimate 29,801 (95% interval 28,451 to 31,151)


In [13]:
# Celda 12: How the model's classes compare with the lexicon in the sample
pd.crosstab(sample["lexicon_class"], sample["llm_class"])

llm_class,not_pastoral,other_pastoral,spiritual
lexicon_class,,,
none,2353,98,49
other_pastoral,358,834,58
spiritual,377,128,745
